# Week 12 Lab 

**Lab Objectives**:
- Gain practical experience with **divide-and-conquer** strategy by solving problems beyond sorting.  
- Implement **Karatsuba multiplication** for large integer arithmetic.  
- Apply **memoisation** for complex dynamic programming problems.  
- Develop **modular and object-oriented code** for algorithm design.  

## Question 1: Karatsuba Multiplication (Divide & Conquer)
**Objective**: Implement a divide-and-conquer based integer multiplication algorithm.  

Karatsuba’s algorithm multiplies two large integers faster than the standard \(O(n^2)\) approach. It is especially useful when numbers have hundreds or thousands of digits (e.g., cryptography, scientific computing).

### Instructions:
1. Implement a **class `KaratsubaMultiplier`** with methods to:
   - Handle multiplication of two integers represented as strings.
   - Recursively divide the problem into subproblems.
   - Combine the results using the Karatsuba formula:  
     \[
     xy = 10^{2m}ac + 10^m(ad + bc) + bd
     \]
   - Where \(a, b\) are parts of \(x\) and \(c, d\) are parts of \(y\).
2. Compare the execution time of:
   - Your Karatsuba implementation.
   - Python’s built-in integer multiplication (`*`).
3. Test with **1000+ digit numbers** and discuss the results.  

In [ ]:
class KaratsubaMultiplier:
    def multiply(self, x: str, y: str) -> str:
        """
        Perform Karatsuba multiplication on two integers represented as strings.
        Return the product as a string.
        """
        x_int, y_int = int(x), int(y)
        if x_int < 10 or y_int < 10:
            return str(x_int * y_int)
        
        max_len = max(len(x), len(y))
        if max_len % 2 != 0:
            max_len += 1
        x = x.zfill(max_len)
        y = y.zfill(max_len)
        
        n = max_len
        m = n // 2
        
        a, b = int(x[:-m]), int(x[-m:])
        c, d = int(y[:-m]), int(y[-m:])

        ac = int(self.multiply(str(a), str(c)))
        bd = int(self.multiply(str(b), str(d)))
        ad_plus_bc = int(self.multiply(str(a + b), str(c + d))) - ac - bd
        
        result = (10 ** (2 * m)) * ac + (10 ** m) * ad_plus_bc + bd
        return str(result)

## Question 2: Memoised Coin Change (DP + Memoisation)

**Objective:** Solve the classic coin-change problem using memoisation.

Given a set of coin denominations and a target amount, compute:

1. The minimum number of coins needed.
2. The number of distinct ways to make the target.

### Instructions:

1. Implement a class CoinChange with:
    - A memoised recursive method to compute the minimum coins.
    - A memoised recursive method to count distinct combinations.
    - An option to trace recursive calls (to show memoisation in action).
2. Test with large target values (e.g., 5000+) and multiple coin denominations.
3. Compare the performance with:
    - Pure recursive implementation.
    - Memoised recursive implementation.
4. Analyse time complexity and space complexity in markdown after running experiments.

In [ ]:
class CoinChange:
    def __init__(self, coins):
        self.coins = coins
        self.memo_min = {}
        self.memo_count = {}
    
    def min_coins(self, target: int) -> int:
        """
        Return the minimum number of coins required to make the target.
        Use memoisation to avoid recomputation.
        """
        if target == 0:
            return 0
        if target < 0:
            return float("inf")
        if target in self.memo_min:
            return self.memo_min[target]
        
        min_val = float("inf")
        for coin in self.coins:
            res = 1 + self.min_coins(target - coin)
            if res < min_val:
                min_val = res
        
        self.memo_min[target] = min_val
        return min_val
    
    def count_ways(self, target: int) -> int:
        """
        Return the total number of distinct ways to make the target.
        Use memoisation to optimise recursive calls.
        """
        if target == 0:
            return 1
        if target < 0:
            return 0
        if target in self.memo_count:
            return self.memo_count[target]
        
        total = 0
        for coin in self.coins:
            total += self.count_ways(target - coin)
        
        self.memo_count[target] = total
        return total

## Question 3: Hybrid Challenge – Karatsuba + Coin Change

**Objective:** Combine divide-and-conquer and memoisation in a practical applied problem.

Suppose we are in a cryptographic setting:
- You are given two large prime numbers p and q (1000+ digits each).
- First, use Karatsuba multiplication to compute N = p * q (simulating RSA modulus generation).
- Then, you are tasked with using coin change to:
    - Find the minimum number of primes needed to express N (coin denominations = first 100 primes).
    - Count the number of possible representations of N using these primes.

### Instructions:

1. Create a class CryptoChallenge that:
    - Uses KaratsubaMultiplier for the multiplication step.
    - Uses CoinChange for the prime decomposition step (treat primes as coin denominations).
2. Since N will be very large, you may restrict the target to a smaller modular subset (e.g., N % 1000) for feasibility.
3. Record and discuss:
    - Execution time.
    - Limitations of the approach.
    - Insights into why divide-and-conquer + memoisation are essential in cryptography and number theory.

In [ ]:
import sympy

class CryptoChallenge:
    def __init__(self, p: str, q: str):
        self.p = p
        self.q = q
        self.multiplier = KaratsubaMultiplier()
        self.product = None
    
    def generate_modulus(self):
        """
        Multiply p and q using Karatsuba to generate N.
        """
        self.product = int(self.multiplier.multiply(self.p, self.q))
        return self.product
    
    def prime_decomposition_coin_change(self, modulus_subset: int) -> dict:
        """
        Use CoinChange on modulus_subset with first 100 primes as denominations.
        Return dictionary with:
          - minimum coins
          - number of ways
        """
        primes = list(sympy.primerange(2, 542))  
        cc = CoinChange(primes)
        
        min_coins = cc.min_coins(modulus_subset)
        num_ways = cc.count_ways(modulus_subset)
        
        return {
            "min_coins": min_coins if min_coins != float("inf") else None,
            "num_ways": num_ways
        }